In [ ]:
dbutils.widgets.text("catalog_param", "my_assessment")
dbutils.widgets.text("schema_param", "gold")

catalog = dbutils.widgets.get("catalog_param")
schema = dbutils.widgets.get("schema_param")

In [ ]:
from pyspark.sql.functions import col

# List of tables to clean
tables_to_clean = ["orders", "lineitem", "customer", "part", "supplier", "nation"]

def clean_bronze_to_silver(table_name):
    # 1. Read from Bronze
    df = spark.table(f"my_assessment.bronze.{table_name}")
    
    # 2. Cleaning Logic
    # .dropDuplicates() removes exact row replicas
    # .dropna() removes rows that are entirely empty
    df_clean = df.dropDuplicates().dropna()
    
    # 3. Drop ingestion metadata (not needed in Silver/Gold)
    df_clean = df_clean.drop("ingestion_date", "source_path")
    
    # 4. Write to Silver Schema
    # Note: We add '_cleaned' to the name to distinguish it
    df_clean.write.mode("overwrite").saveAsTable(f"my_assessment.silver.{table_name}_cleaned")
    
    print(f"✅ Table {table_name} cleaned and moved to Silver.")

# Run the cleaning function for all tables
for t in tables_to_clean:
    clean_bronze_to_silver(t)